# ResNet18 ERPgnostics training export

This notebook trains the binary ERP-pattern ResNet18 on the real JLD2 datasets
under `datasets/` and writes a reusable model artifact. It intentionally does
not build the ERPgnostics explorer; use
`resnet18_erpgnostics_import_explorer.ipynb` for visualization.



In [1]:
# Optional quick-run overrides. Uncomment before running this first cell.
# ENV["WEEK24_RUN_CV"] = "false"
# ENV["WEEK24_RESNET18_EPOCHS"] = "1"

include(joinpath(@__DIR__, "resnet18_erpgnostics_common.jl"))

TRAIN_EXPORT_DIR = joinpath(NOTEBOOK_DIR, "outputs", "resnet18_erpgnostics_train_export")
TRAIN_MODEL_ARTIFACT = model_artifact_path(TRAIN_EXPORT_DIR)

println("Training output directory: ", TRAIN_EXPORT_DIR)
println("Model artifact path: ", TRAIN_MODEL_ARTIFACT)
println("TARGET_TRIALS = ", TARGET_TRIALS)



REPO_ROOT = /home/benjamin/Dokumente/BA2/
DATASETS_ROOT = /home/benjamin/Dokumente/BA2/datasets
Training output directory: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export
Model artifact path: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export/resnet18_final_state.jld2
TARGET_TRIALS = 200


## Train

By default this runs the same 5-fold CV plus final all-data model as the
combined notebook. For a quick smoke run, set these before executing:

```julia
ENV["WEEK24_RUN_CV"] = "false"
ENV["WEEK24_RESNET18_EPOCHS"] = "1"
```



In [2]:
training_run = run_training_pipeline(
    output_dir = TRAIN_EXPORT_DIR,
    nepochs = TRAIN_EPOCHS,
    lr = TRAIN_LR,
    k_folds = K_FOLDS,
    seed = GLOBAL_SEED,
    run_cv = RUN_CV,
)



Loading real JLD2 labels from /home/benjamin/Dokumente/BA2/datasets.
Materializing fixed-trial ERP images with inverse-sort/polarity variants.
Training and validating ResNet18 with 5-fold CV.
CUDA device: NVIDIA GeForce RTX 4070
resnet18_real_jld2_inverse_sort_polarity_binary | CV fold 1/5 | train=14876 | val=3720
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 1/8 | loss=0.50475
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 2/8 | loss=0.28275
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 3/8 | loss=0.20926
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 4/8 | loss=0.16014
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 5/8 | loss=0.13916
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 6/8 | loss=0.09994
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 7/8 | loss=0.0936
resnet18_real_jld2_inverse_sort_polarity_binary_fold1 | epoch 8/8 | loss=0.08148
resnet18_real_jld2_inverse_sort_polar

(labels_df = 2869×9 DataFrame
  Row │ dataset_key                dataset_label                      channel_ ⋯
      │ String                     String                             String   ⋯
──────┼─────────────────────────────────────────────────────────────────────────
    1 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E1       ⋯
    2 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E111
    3 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E121
    4 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E51
    5 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E52      ⋯
    6 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E59
    7 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E70
    8 │ 02_new_eegeyenet_saccades  EEGEyeNet minimally processed sa…  E72
  ⋮   │             ⋮                              ⋮                       ⋮   ⋱
 2863 │ nod_eeg_public             NOD

## Export Model

The artifact stores the full CPU `Flux.state(model)` plus metadata. This is
important for ResNet18 because BatchNorm running statistics are model state,
not trainable parameters. Saving only `Flux.trainables(model)` makes the
imported model produce nearly constant scores.



In [3]:
artifact_metadata = Dict{String, Any}(
    "source_notebook" => "notebooks/week_24/resnet18_erpgnostics_train_export.ipynb",
    "output_dir" => TRAIN_EXPORT_DIR,
    "run_config_path" => joinpath(TRAIN_EXPORT_DIR, "run_config.json"),
    "final_train_metrics_path" => joinpath(TRAIN_EXPORT_DIR, "final_train_metrics.csv"),
    "fold_metrics_path" => joinpath(TRAIN_EXPORT_DIR, "fold_metrics.csv"),
    "trained_on" => "real JLD2 datasets from datasets/, excluding datasets/simulated",
)

save_resnet18_model_artifact(TRAIN_MODEL_ARTIFACT, training_run.final_model; metadata = artifact_metadata)
println("Saved model artifact: ", TRAIN_MODEL_ARTIFACT)

training_run.final.metrics_df



Saved model artifact: /home/benjamin/Dokumente/BA2/notebooks/week_24/outputs/resnet18_erpgnostics_train_export/resnet18_final_state.jld2


Row,model_name,n_train,train_accuracy,train_balanced_accuracy,train_macro_f1,train_precision,train_recall,train_time_s,pretrained_params_loaded,batchsize,use_cuda
,String,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Int64,Int64,Bool
1,resnet18_real_jld2_inverse_sort_polarity_binary_final,18596,0.982631,0.983077,0.982449,0.981892,0.983077,35.9697,62,32,true


## Training Summary



In [4]:
if nrow(training_run.cv.metrics_df) > 0
    summarize_metrics(training_run.cv.metrics_df)
else
    training_run.final.metrics_df
end


Row,model_name,val_accuracy_mean,val_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,val_macro_f1_mean,val_macro_f1_std,val_precision_mean,val_recall_mean,train_time_mean_s,pretrained_params_loaded
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64
1,resnet18_real_jld2_inverse_sort_polarity_binary,0.910788,0.0084102,0.909909,0.00632657,0.909721,0.00808947,0.91111,0.909909,37.2215,62
